# Notebook 1: Data Exploration

Initial exploration of the SF Police Incident Reports dataset.  
Uses **pandas** locally — no Spark required.

Dataset: `data/sf_incidents.csv` (796K rows, 35 columns)

In [17]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/sf_incidents.csv"

# Load a sample first to inspect schema quickly
sample = pd.read_csv(DATA_PATH, nrows=5000, low_memory=False)
print(f"Sample shape: {sample.shape}")
sample.head(3)

Sample shape: (5000, 35)


,Incident Datetime,Incident Date,Incident Time,Incident Year,Incident Day of Week,Report Datetime,Row ID,Incident ID,Incident Number,CAD Number,...,Longitude,Point,Neighborhoods,ESNCAG - Boundary File,Central Market/Tenderloin Boundary Polygon - Updated,Civic Center Harm Reduction Project Boundary,HSOC Zones as of 2018-06-05,Invest In Neighborhoods (IIN) Areas,Current Supervisor Districts,Current Police Districts
0,2023/03/13 11:41:00 PM,2023/03/13,23:41,2023,Monday,2023/03/13 11:41:00 PM,125373607041,1253736,230167874,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023/03/01 05:02:00 AM,2023/03/01,05:02,2023,Wednesday,2023/03/11 03:40:00 PM,125379506374,1253795,236046151,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023/03/13 01:16:00 PM,2023/03/13,13:16,2023,Monday,2023/03/13 01:17:00 PM,125357107041,1253571,220343896,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Column overview
print("Columns:")
for i, col in enumerate(sample.columns, 1):
    print(f"  {i:2}. {col}")

Columns:
   1. Incident Datetime
   2. Incident Date
   3. Incident Time
   4. Incident Year
   5. Incident Day of Week
   6. Report Datetime
   7. Row ID
   8. Incident ID
   9. Incident Number
  10. CAD Number
  11. Report Type Code
  12. Report Type Description
  13. Filed Online
  14. Incident Code
  15. Incident Category
  16. Incident Subcategory
  17. Incident Description
  18. Resolution
  19. Intersection
  20. CNN
  21. Police District
  22. Analysis Neighborhood
  23. Supervisor District
  24. Supervisor District 2012
  25. Latitude
  26. Longitude
  27. Point
  28. Neighborhoods
  29. ESNCAG - Boundary File
  30. Central Market/Tenderloin Boundary Polygon - Updated
  31. Civic Center Harm Reduction Project Boundary
  32. HSOC Zones as of 2018-06-05
  33. Invest In Neighborhoods (IIN) Areas
  34. Current Supervisor Districts
  35. Current Police Districts


In [19]:
# Load full dataset
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Full dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Full dataset shape: (796456, 35)
Memory usage: 412.9 MB


In [20]:
# Data types and null counts
info = pd.DataFrame({
    "dtype": df.dtypes,
    "nulls": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(2),
    "unique": df.nunique()
})
info

,dtype,nulls,null_pct,unique
Incident Datetime,str,0,0.00,375986
Incident Date,str,0,0.00,2154
Incident Time,str,0,0.00,1440
Incident Year,int64,0,0.00,6
Incident Day of Week,str,0,0.00,7
Report Datetime,str,0,0.00,567733
Row ID,int64,0,0.00,796456
Incident ID,int64,0,0.00,666545
Incident Number,int64,0,0.00,575466
CAD Number,float64,179498,22.54,455378


In [21]:
# Parse datetime column
df["Incident Datetime"] = pd.to_datetime(df["Incident Datetime"], format="%Y/%m/%d %I:%M:%S %p", errors="coerce")
df["hour"] = df["Incident Datetime"].dt.hour
df["month"] = df["Incident Datetime"].dt.month
df["year"] = df["Incident Datetime"].dt.year

bad = df["Incident Datetime"].isna().sum()
print(f"Unparseable datetime rows (set to NaT): {bad}")
print("Year range:", df["year"].min(), "–", df["year"].max())
print("Date range:", df["Incident Datetime"].min(), "→", df["Incident Datetime"].max())

Unparseable datetime rows (set to NaT): 1
Year range: 2018.0 – 2023.0
Date range: 2018-01-01 00:00:00 → 2023-11-24 22:30:00


In [22]:
# Target variable distribution
cat_counts = df["Incident Category"].value_counts()
print(f"Unique categories: {cat_counts.shape[0]}")
print("\nTop 15:")
cat_counts.head(15)

Unique categories: 49

Top 15:


Incident Category
Larceny Theft          242034
Other Miscellaneous     54225
Malicious Mischief      53860
Assault                 48501
Non-Criminal            46987
Burglary                44390
Motor Vehicle Theft     42555
Recovered Vehicle       31927
Fraud                   25926
Lost Property           23194
Warrant                 22623
Drug Offense            19604
Robbery                 17916
Missing Person          17176
Suspicious Occ          16323
Name: count, dtype: int64

In [23]:
# Police district distribution
df["Police District"].value_counts(dropna=False)

Police District
Central       118242
Northern      107897
Mission        98785
Southern       95225
Tenderloin     77355
Bayview        70894
Ingleside      61130
Taraval        56321
Richmond       49818
Park           36453
Out of SF      24336
Name: count, dtype: int64

In [24]:
# Geo bounds — sanity check
geo = df[["Latitude", "Longitude"]].dropna()
print(f"Latitude:  {geo['Latitude'].min():.4f} – {geo['Latitude'].max():.4f}")
print(f"Longitude: {geo['Longitude'].min():.4f} – {geo['Longitude'].max():.4f}")
print(f"Rows with coordinates: {len(geo):,} / {len(df):,} ({len(geo)/len(df)*100:.1f}%)")

Latitude:  37.7080 – 37.8300
Longitude: -122.5113 – -122.3637
Rows with coordinates: 753,186 / 796,456 (94.6%)


In [25]:
# Resolution values
df["Resolution"].value_counts(dropna=False)

Resolution
Open or Active          641682
Cite or Arrest Adult    148564
Unfounded                 4094
Exceptional Adult         2116
Name: count, dtype: int64

In [26]:
# Records per year
df.groupby("year").size().rename("incident_count")

year
2018.0    151673
2019.0    146839
2020.0    117474
2021.0    128236
2022.0    135292
2023.0    116941
Name: incident_count, dtype: int64

In [27]:
# Missing values in ML features
ml_cols = ["Incident Datetime", "Latitude", "Longitude",
           "Police District", "Incident Category"]
print("Missing values in ML feature columns:")
df[ml_cols].isnull().sum()

Missing values in ML feature columns:


Incident Datetime        1
Latitude             43270
Longitude            43270
Police District          0
Incident Category      713
dtype: int64